# Making a Simple MMMAudio Graph

MMMAudio works with two languages: Mojo for DSP and Python for control/interaction. The .mojo file that runs your DSP code is called the *graph*. This tutorial will walk you through creating a simple graph step by step, and controlling it through python.


## Anatomy of a MMMAudio Graph

An MMMAudio Graph is made of a Mojo struct (think like a class in Python or other programming languages. This isn't exactly 1-to-1 but will get you in the right ballpark). There are a number of required fields for a MMMAudio Graph to run properly:

* Declaration - The name of your graph and its Traits (this is a Mojo thing, they're important but outside of the scope of this tutorial to talk about here).
* Struct variables - Data and structs that will stored to be used in your graph.
* \__init__() - Initializes the struct variables to default values.
* next() - The code that is ran when a sample is processed.

An important note, the name of your graph *must* be the same as the file that it is stored in, capitalization and all. Let's break down the parts:

### Declaration

First, we need to make a file to store our graph in. You can place this in the "user_graphs" directory found in this repo. You can find a completed version of this graph in the "demo_graphs" directory as well.

Now that we've made our .mojo file, we'll need to import mmm_audio:

```mojo
from mmm_audio import *
```

Then, declare a new struct:

```mojo
struct MyFirstGraph(Movable, Copyable):
```

Note that after the ```struct``` keyword and name, inside the parentheses are the Traits ```Movable``` and ```Copyable```. These are required for your Graph to work. Also note that the name of the struct, MyFirstGraph, is the exact same as the name of the file without the .mojo file extension.

### Struct Variables

After the declaration is a list of ```struct``` variables. These can be accessed inside of the graph by calling ```self.variable_name```. For this example, we'll keep it simple and have three variables: a synth variable that will hold an ```Osc[1]``` struct, a freq variable to store the frequency for our oscillator, and vol for the volume of our oscillator:

```mojo
struct MyFirstGraph(Movable, Copyable):
    var synth: Osc[1]
    var freq: Float64
    var vol: Float64
```
Notice that there are no values assigned to these variables, only the data type that is held in them.

### \__init__()

The \__init__() method of the Graph sets the default values for our Graph's variables and creates instances of any structs we might use (such as the ```Osc[1]``` stored in synth). 
All struct variables *must* be given a default value in the \__init__() method. \__init__() also has two required arguments: out self, and world: World. Notice that method arguments require a *type* designation.

```mojo

struct MyFirstGraph(Movable, Copyable):
    var synth: Osc[1]
    var freq: Float64
    var vol: Float64
    
    def __init__(out self, world: World):
        self.synth = Osc[1](world)
        self.freq = 440.0
        self.vol = 0.25
        
```

### next()

The ```next()``` method is the code that is ran each sample for the Graph. This is where the struct variables are used. ```next()``` is a bit more involved, so let's take it a step at a time. First, we must make a declaration for next, this has two parts: arguments and SIMD out size. ```next()``` only requires one argument, "mut self", though you can add arguments for more complex graphs. The SIMD out size determines the number of *output channels* the Graph has. This is defined by an ```MFloat[x]``` where ```x``` is the number of output channels. Importantly, ```x``` *must* be a power of two. If your speaker array is less than a power of two, ```x``` must be the power of two *above* the number of your speakers (ie. 5 speakers -> ```MFloat[8]```). Our Graph will have two output channels:

```mojo
#        arguments
def next(mut self   ) -> MFloat[2]:
        
```

The ```-> MFloat[2]``` means that the ```next()``` method *must* ```return``` a value, in this case an ```MFloat[2]```. It's good practice to create an ```out``` variable in your ```next()``` method to hold your outgoing signal and ```return``` that. Below ```out``` has been initialized to an ```MFloat[2]``` with all zero values ( [0.0, 0.0] ):


```mojo
def next(mut self) -> MFloat[2]:
        var out = MFloat[2](0.0)

        return out
```

MMMAudio uses *composition*, meaning that all MMMAudio structs are just graphs themselves. Therefore, each MMMAudio struct (such as ```Osc[1]```) have ```next()``` methods themselves that return an ```MFloat[x]```. ```Osc[1]``` outputs a monophonic signal, while our Graph expects a stereo signal. If we try to assign the output of ```Osc[1]``` directly to ```out``` we'll get an error:

```mojo
def next(mut self) -> MFloat[2]:
        var out = self.synth.next() ❌ # This doesn't work
        
        return out

```

Instead we can do one of two things, either assign the output of our oscillator to each index of our ```out``` ```MFloat```:

```mojo
def next(mut self) -> MFloat[2]:
    var osc = self.synth.next()
    var out = MFloat[2](0.0)
    
    out[0] = osc
    out[1] = osc

    return out ✅ 
```

Or increment the entire ```out``` ```MFloat``` by the output of our ```Osc[1]```:

```mojo
def next(mut self) -> MFloat[2]:
        var out = MFloat[2](0.0)
        out += self.synth.next()
        return out ✅
```

Let's add our other parameters. Many MMMAudio structs take arguments in their ```next()``` method. ```Osc[1]``` has three possible arguments, though not every one needs to be given:

```mojo
    Osc.next(freq: Float64, phase: Float64, trig: Bool)
```

We'll add the struct property ```self.freq``` as the first argument so we can control the pitch of the oscillator:

```mojo
    def next(mut self) -> MFloat[2]:
        var out = MFloat[2](0.0)
        out += self.synth.next(self.freq)
        return out
```

We can then multiply the out by the struct property ```self.vol``` to scale the output volume:

```mojo
    def next(mut self) -> MFloat[2]:
        var out = MFloat[2](0.0)
        out += self.synth.next(self.freq)
        return out * self.vol
```



In [ ]:
# This code only needs to be executed in a Jupyter Notebook and otherwise would not be present in a standard MMMAudio Python file.
import os
# Move up one level to the parent directory
os.chdir('..') 
print(os.getcwd())

In [ ]:
# Import MMMAudio Python library
from mmm_python import *

# Create an MMMAudio instance and store it in the variable mmm_audio. You can change the graph being loaded by changing the graph name and package name (the package name is the folder where the .mojo file is located)
# If you placed your .mojo file in the "user_graphs" folder you will need to change the package_name argument to: "mmm_audio_tutorials.user_graphs"
mmm_audio = MMMAudio(256, num_output_channels=2, graph_name="MyFirstGraph", package_name="mmm_audio_tutorials.demo_graphs")

In [ ]:
mmm_audio.start_audio()

In [ ]:
mmm_audio.stop_audio()

You'll notice that if you try to change values in your .mojo file while the synth is running nothing will happen. This is because Mojo is a *compiled* language, where we create a static program from an input file. MMMAudio addresses this through its Messenger system, which allows Python and Mojo to communicate. 

## Messaging

In order to receive messages from Python we need to add a ```Messenger``` struct to our Graph:

```mojo

struct MyFirstGraph(Movable, Copyable):
    var synth: Osc[1]
    var freq: Float64
    var vol: Float64
    var messenger: Messenger #<--- Adding a struct property to hold the messenger
    
    def __init__(out self, world: World):
        self.synth = Osc[1](world)
        self.messenger = Messenger(world) #<--- Instantiating our messenger
        self.freq = 440.0
        self.vol = 0.25
    
    def next(mut self) -> MFloat[2]:
        var out = MFloat[2](0.0)
        out += self.synth.next(self.freq)
        return out * self.vol

```

We can then handle incoming messages in the Graph's ```next()``` method. The simplest way to do this is with ```Messenger```'s ```update()``` method. ```update()``` takes two arguments: an address that the messages are sent to (as a string), and a variable to store the incoming value in.

We'll want to be able to change the frequency and volume of our oscillator, so we'll call the ```update()``` method twice:

```mojo
struct MyFirstGraph(Movable, Copyable):
    var synth: Osc[1]
    var freq: Float64
    var vol: Float64
    var messenger: Messenger
    
    def __init__(out self, world: World):
        self.synth = Osc[1](world)
        self.messenger = Messenger(world)
        self.freq = 440.0
        self.vol = 0.25
    
    def next(mut self) -> MFloat[2]:
        var out = MFloat[2](0.0)
        self.messenger.update("freq", self.freq) #<-- Updates the value stored in sef.freq
        self.messenger.update("vol", self.vol) #<-- Updates the value stored in self.vol
        out += self.synth.next(self.freq)
        return out * self.vol

```

Then we can send messages to Mojo from Python by calling the ```send_float()``` method on our MMMAudio instance, providing the address to send to and value:

In [ ]:
# Update the self.freq property
mmm_audio.send_float("freq", 880)

In [ ]:
# Update the self.vol property
mmm_audio.send_float("vol", 0.0)

In [ ]:
# Update the self.freq and self.vol properties at the same time
mmm_audio.send_float("freq", 660)
mmm_audio.send_float("vol", 0.125)

Congrats! You've succesful created your first MMMAudio program! 